In [ ]:
# | default_exp features.endmotif_1mer

In [ ]:
# | export
from __future__ import annotations
import pandas as pd
import structlog
from kreview.eval_engine import FeatureEvaluator

log = structlog.get_logger()

In [ ]:
# | export


class EndMotif1merEvaluator(FeatureEvaluator):
    """Extracts 1-mer fragment end base frequencies with strand bias metrics.

    Derived metrics:
    - Purine/pyrimidine asymmetry: (A+G) - (C+T)
    - A/T strand bias: A / (A+T)
    - C/G strand bias: C / (C+G)
    """

    name = "EndMotif1mer"
    source_file = ".EndMotif1mer.parquet"
    tier = 3
    category = "motifs"

    def extract(self, df: pd.DataFrame) -> dict[str, float]:
        extracted = {}
        try:
            if df.empty:
                return extracted
            cols = set(df.columns)

            if "base" in cols and "fraction" in cols:
                for row in df.to_dict("records"):
                    b = str(row["base"])
                    if pd.notna(row["fraction"]):
                        extracted[f"base_1mer_{b}"] = float(row["fraction"])

            # --- Derived: strand bias metrics ---
            a = extracted.get("base_1mer_A", 0.0)
            t = extracted.get("base_1mer_T", 0.0)
            c = extracted.get("base_1mer_C", 0.0)
            g = extracted.get("base_1mer_G", 0.0)
            total = a + t + c + g

            if total > 0:
                extracted["purine_pyrimidine_asymmetry"] = float((a + g) - (c + t))
                if (a + t) > 0:
                    extracted["at_strand_bias"] = float(a / (a + t))
                if (c + g) > 0:
                    extracted["cg_strand_bias"] = float(c / (c + g))

            return extracted
        except Exception as e:
            log.exception("extraction_failed", evaluator=self.name, error=str(e))
            return {}

In [ ]:
# | test
evaluator = EndMotif1merEvaluator()
df = pd.DataFrame([{"base": "A", "fraction": 0.25}])
res = evaluator.extract(df)
assert res["base_1mer_A"] == 0.25